# Udeco 形态边沿与级联状态分解（单样本交互实验）

本 Notebook 用于验证“**边沿形态先判接触模态，再在 Contact 内分解压力与慢基线**”的方案。
Udeco2 与 Udeco3 使用完全相同的函数接口，但各自独立估计噪声、候选边沿和状态。

物理分量约定：

- $J(t)$：Contact / Non-contact 模态差以及 attach / detach 过渡；恢复 Contact 后不保留累计偏置。
- $P(t)$：只在接触模态内解释的压迫/松弛中尺度分量。
- $B_{slow}(t)$：只在真实、稳定 Contact 点缓慢更新的长期基线。
- residual：为保证恒等式闭合而保留的剩余高频项。

主线为：

```text
Raw → 3 点 Median → 10 s Bcontext → 多尺度 Haar 边沿
    → 形态分类 → Contact/Non-contact 状态机 → Ucontact
    → Robust LOWESS Bm_contact → Conditional Bslow → P / J
```

In [ ]:
# ================================================================
# 样本与协议
# ================================================================
# 只修改 SAMPLE_NAME 即可切换样本；程序会唯一匹配 */<样本名>_sensor.csv。
SAMPLE_NAME = "cgx_rope_1"

# 数据根目录。通常无需修改；若数据移动，只改此处即可。
DATA_ROOT = "subject_data"

# 协议采样率，单位 Hz。当前仓库数据为 100 Hz；它决定所有秒制参数到采样点的换算。
# 增大/减小仅应反映真实采样协议，不应用它调算法；推荐保持 100。
FS = 100

# ================================================================
# 预处理：短中值滤波
# ================================================================
# 中值窗点数：仅去掉孤立尖峰，默认 3 点奇数窗更少改变 1–4 点快速边沿。
# 增大：抑制尖峰更强，但可能抹平短暂非接触；减小：边沿更原始，但单点噪声更易触发候选。
# 单位：采样点；推荐 1–3 点，3 点为当前默认。
MEDIAN_FILTER_POINTS = 3

# ================================================================
# 检测上下文 Bcontext：只服务于边沿定位，不进入最终分解
# ================================================================
# LOWESS 上下文窗口，单位秒。它描述比 attach/detach 更低频的局部参考。
# 增大：保留更多快速/中速变化在 edge_signal 中；减小：上下文更贴信号，可能吃掉慢 attach。
# 推荐 6–15 s；默认 10 s。
CONTEXT_WINDOW_SEC = 10.0

# LOWESS 稳健重加权次数。增大：更不受离群点影响但更慢；减小：更快但受强边沿牵引更多。
# 单位：次；推荐 1–2，默认 1 保持初步趋势简单。
CONTEXT_ROBUST_ITERS = 1

# ================================================================
# 多尺度 Haar 定位与 L/R 平台测量
# ================================================================
# 候选时间尺度，单位秒。短尺度定位 1–4 点 detach，长尺度覆盖慢 attach。
# 增加长尺度：可找到更慢恢复，但更易响应压力坡；删除短尺度：会漏掉陡峭 detach。
# 推荐覆盖约 0.02–2.56 s，按约 2 倍递增。
EDGE_SCALES_SEC = (0.02, 0.04, 0.08, 0.16, 0.32, 0.64, 1.28, 2.56)

# 多尺度同方向响应的软门限：至少两个相邻尺度达到此值即可成为候选。
# 增大：弱边沿候选更少；减小：漏检减少但需形态层排除更多噪声。无量纲，推荐 2.5–4。
EDGE_Z_THRESHOLD = 3.0

# 单一尺度须达到此较高门限。增大：更保守；减小：会保留更多单尺度噪声。
# 无量纲，推荐 4–6；默认 4 保留原先强单尺度触发能力。
EDGE_SINGLE_SCALE_Z_THRESHOLD = 4.0

# 同方向相邻尺度支持数。增大：要求更一致；减小：更敏感但更易受偶然峰影响。
# 单位：尺度数；推荐 2–3，默认 2；强单尺度仍可按上一门限触发。
EDGE_MIN_SCALE_SUPPORT = 2

# 相邻响应峰合并距离，单位秒。增大：同一边沿更不易重复；减小：近邻事件更容易分开。
# 推荐 0.04–0.15 s。
EDGE_MERGE_SEC = 0.08

# L/R 平台固定短窗的最短/最长长度，单位秒。候选尺度会在这两个边界内自适应选窗。
# 窗口增大：平台中位数更稳但更可能混入压力趋势；减小：更局部但噪声更大。
# 100 Hz 下推荐 0.04–0.20 s，即 4–20 点。
EDGE_LEVEL_WINDOW_MIN_SEC = 0.04
EDGE_LEVEL_WINDOW_MAX_SEC = 0.20

# 平台窗与候选边沿之间的避让区，单位秒。增大：L/R 更少受过渡污染；减小：适合极短事件。
# 推荐 0.01–0.05 s。
EDGE_GUARD_SEC = 0.02

# 平台容差中的噪声倍率。增大：更易认定已进入平台，边界可能内缩；减小：边界外扩但更受噪声干扰。
# 无量纲；推荐 0.5–1.5，默认 1.0，噪声由局部 MAD 换算为标准差。
EDGE_PLATFORM_TOLERANCE_SIGMA = 1.0

# 平台容差占最终 L/R 差值的最小比例。增大：容忍更大波动；减小：更贴近理想 0%/100% 边界。
# 无量纲；推荐 0.02–0.08，默认 0.03。
EDGE_PLATFORM_TOLERANCE_FRACTION = 0.03

# 离开 L / 稳定进入 R 的连续点数。增大：更抗噪但短事件易漏；减小：更敏感但噪声易被当平台。
# 单位：采样点；推荐 2–3，默认 2，兼顾短暂非接触。
EDGE_PLATFORM_STABLE_POINTS = 2

# ================================================================
# 边沿形态分类：detach 快速上升，attach 较慢下降
# ================================================================
# 当前传感器物理方向约定。若未来硬件极性相反，应同时交换这两项，而不是单独改阈值。
DETACH_DIRECTION = "up"
ATTACH_DIRECTION = "down"

# detach 的完整平台边界最大宽度，单位采样点。增大：允许较钝的上升也算脱离；
# 减小：更强调瞬时断开，但可能漏掉受噪声影响的真边沿。推荐 4–8 点，默认 6。
DETACH_MAX_POINTS = 6

# detach 平台差相对局部噪声的最小 SNR。增大：排除噪声更强；减小：能检出弱脱离但误报增多。
# 单位：无量纲；推荐 4–8。
DETACH_MIN_SNR = 5.0

# detach 幅度占本路全段稳健动态范围的最小比例。增大：只保留显著模态切换；
# 减小：可检出小幅脱离，但更易把局部抖动当 detach。推荐 0.02–0.08。
DETACH_MIN_RELATIVE_AMPLITUDE = 0.03

# detach 前平台相对全段低/高稳健分位的位置下限。数值越大越要求从低模态起跳；
# 数值越小越包容慢漂移造成的平台移动。推荐 -0.3–0.1。
DETACH_PRE_LEVEL_FRACTION = -0.20

# attach 完整平台边界最小宽度，单位采样点。增大：更强调接触建立较慢；减小：允许更快恢复。
# 推荐 4–10 点，默认 4；与前次 detach 的宽度比另由状态机检查。
ATTACH_MIN_POINTS = 4

# attach 过渡段方向一致率。增大：只接受更单调下降；减小：容忍振铃但压力变化误报会增加。
# 单位 0–1；推荐 0.55–0.80。
ATTACH_MIN_MONOTONICITY = 0.65

# attach 平台差的最小 SNR。增大更保守，减小可检出弱恢复；推荐 3–7。
ATTACH_MIN_SNR = 4.0

# attach 幅度占本路动态范围的最小比例。增大排除小波动，减小保留弱恢复；推荐 0.005–0.05。
ATTACH_MIN_RELATIVE_AMPLITUDE = 0.01

# 新平台持续性下限。增大：要求边沿后更稳定；减小：允许短暂非接触/接触段。
# 单位 0–1；推荐 0.5–0.8。
EDGE_MIN_PERSISTENCE = 0.60

# 当 attach 前已有 detach 时，attach 宽度至少为 detach 宽度的倍数。
# 增大：快断慢接的物理先验更强；减小：允许两边沿速度相近。推荐 1.5–3.0。
EDGE_ASYMMETRY_RATIO = 1.5

# 判断 L/R 是否属于两个模态的噪声倍率。增大：要求平台分离更明显；减小：接受短弱非接触。
# 推荐 1.5–4.0。
MODE_LEVEL_NOISE_RATIO = 2.0

# 模态平台差占已观测 detach 幅度的最低比例。增大：拒绝不完整恢复；减小：容忍漂移与短平台。
# 推荐 0.2–0.6。
MODE_LEVEL_DELTA_FRACTION = 0.35

# ================================================================
# 一级状态机与接触模态重建
# ================================================================
# 边沿进入状态机的最低置信度。增大更保守，减小更敏感；推荐 0.45–0.75。
STATE_MIN_CONFIDENCE = 0.55

# 接受事件后的不应期，单位秒。增大可抑制振铃重复触发；减小允许非常短的脱离-恢复。
# 推荐 0.04–0.20 s。
STATE_REFRACTORY_SEC = 0.08

# 被拒绝的模糊候选向两侧扩展的 uncertain 标记，单位秒。增大更谨慎地冻结基线；
# 减小保留更多 Contact 有效点。推荐 0.02–0.15 s。
UNCERTAIN_MARGIN_SEC = 0.05

# Ucontact 插值时左右真实接触锚点窗口，单位秒。增大：锚点更稳；减小：更贴近事件局部。
# 推荐 0.5–2.0 s。
CONTACT_ANCHOR_WINDOW_SEC = 1.0

# ================================================================
# 最终 Bm_contact 与条件 Bslow
# ================================================================
# Ucontact 上最终 Robust LOWESS 窗口，单位秒。增大：Bm 更平滑但压迫缓坡被削弱；
# 减小：保留更快压力变化但也保留更多噪声。推荐 1–6 s。
CONTACT_LOESS_WINDOW_SEC = 3.0

# 最终 LOWESS 稳健迭代次数。增大抗离群更强但更慢；推荐 1–3。
CONTACT_ROBUST_ITERS = 2

# Bslow 时间常数，单位秒。增大：基线更慢、更不吸收压力；减小：更快跟随长期漂移。
# 推荐 30–180 s；若目前 Bslow 变化过小，可先降至 30–45 s。
BSLOW_TAU_SEC = 60.0

# Bslow 更新速率倍率。增大：稳定区更新更快；减小：更保守。推荐 0.5–3.0。
BSLOW_UPDATE_RATE = 1.0

# 允许更新 Bslow 的最大 |dBm/dt|，单位 Udeco/s。增大：更多缓坡被吸收进基线；
# 减小：压力保护更强但 Bslow 容易长期冻结。推荐 1e-4–1e-3。
BSLOW_STABLE_SLOPE_THRESHOLD = 3e-4

# 允许更新时 |Bm-Bslow| 的最大偏移，单位 Udeco。增大：Bslow 可追赶更大漂移；
# 减小：防止压力平台被吸收，但过小会一旦偏离就难以恢复。推荐 5e-4–5e-3。
BSLOW_STABLE_OFFSET_THRESHOLD = 1e-3

# ================================================================
# 二级 Contact 内压力状态与 J
# ================================================================
# 压力 loading/relaxing 的斜率阈值，单位 Udeco/s。增大：更多点归为静态平台；
# 减小：对缓慢压力变化更敏感。推荐 1e-4–1e-3。
PRESSURE_SLOPE_THRESHOLD = 3e-4

# baseline/pressure_hold 的 |P| 分界，单位 Udeco。增大：更多偏移仍视作基线；
# 减小：更容易标为压力保持。推荐 5e-4–5e-3。
PRESSURE_OFFSET_THRESHOLD = 1e-3

# 只对 Non-contact/Transition 内 J 做的短中值平滑窗口，单位秒。
# 增大：J 更平滑但边沿更钝；减小：保留更多高频噪声。推荐 0.05–0.20 s。
J_SMOOTH_WINDOW_SEC = 0.10

# ================================================================
# 交互绘图（只改变显示，不修改算法结果）
# ================================================================
# 初始显示范围，单位秒；设为 None 表示从头/到尾。之后可在 Plotly 中单独框选放大。
PLOT_START_SEC = 70.0
PLOT_END_SEC = 140.0

# True：同一路的六行图联动缩放；两路仍是两个独立 Figure，互不影响。
LINK_X_AXES_WITHIN_CHANNEL = True

# 每张候选表最多显示的行数。增大便于全面排查但输出更长；推荐 40–150。
EDGE_TABLE_MAX_ROWS = 80

## 导入与数据读取

`Bcontext` 只是 10 秒检测上下文；最终 `Bm_contact` 会在状态机完成、非接触区被接触模态重建后重新估计，二者不要混为同一条基线。

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from udeco_analysis import load_udeco_sample
from udeco_cascade_state_decomposition import (
    CascadeStateConfig,
    decompose_cascade_udeco,
)

# plotly_mimetype 会把交互图的 JSON 保存在 Notebook 输出中，重新打开后仍可缩放和悬停。
pio.renderers.default = "plotly_mimetype"

In [ ]:
# 在所有受试者目录下唯一匹配传感器 CSV；因此切换样本只需改首个代码单元的 SAMPLE_NAME。
data_root = Path(DATA_ROOT).resolve()
matches = sorted(data_root.glob(f"*/{SAMPLE_NAME}_sensor.csv"))
if len(matches) != 1:
    raise FileNotFoundError(
        f"期望唯一匹配 */{SAMPLE_NAME}_sensor.csv，实际找到 {len(matches)} 个：{matches}"
    )

loaded = load_udeco_sample(matches[0], data_root=data_root, fs=FS)
time = loaded.time_s
udeco2 = loaded.udeco2
udeco3 = loaded.udeco3
fs = loaded.fs

print(f"样本：{loaded.sample_name}（{loaded.subject}）")
print(f"文件：{matches[0]}")
print(f"采样率：{fs} Hz；Udeco2/Udeco3 长度：{len(udeco2)}/{len(udeco3)}")

## 双通道独立分解

下面只构造一次配置，并对 Udeco2/Udeco3 分别调用同一接口。两次调用之间不共享局部噪声、候选事件、状态或 Bslow。

In [ ]:
config = CascadeStateConfig(
    median_filter_points=MEDIAN_FILTER_POINTS,
    context_window_s=CONTEXT_WINDOW_SEC,
    context_robust_iterations=CONTEXT_ROBUST_ITERS,
    edge_scales_s=EDGE_SCALES_SEC,
    edge_z_threshold=EDGE_Z_THRESHOLD,
    edge_single_scale_z_threshold=EDGE_SINGLE_SCALE_Z_THRESHOLD,
    edge_min_scale_support=EDGE_MIN_SCALE_SUPPORT,
    edge_merge_s=EDGE_MERGE_SEC,
    edge_level_window_min_s=EDGE_LEVEL_WINDOW_MIN_SEC,
    edge_level_window_max_s=EDGE_LEVEL_WINDOW_MAX_SEC,
    edge_guard_s=EDGE_GUARD_SEC,
    edge_platform_tolerance_sigma=EDGE_PLATFORM_TOLERANCE_SIGMA,
    edge_platform_tolerance_fraction=EDGE_PLATFORM_TOLERANCE_FRACTION,
    edge_platform_stable_points=EDGE_PLATFORM_STABLE_POINTS,
    detach_direction=DETACH_DIRECTION,
    detach_max_points=DETACH_MAX_POINTS,
    detach_min_snr=DETACH_MIN_SNR,
    detach_min_relative_amplitude=DETACH_MIN_RELATIVE_AMPLITUDE,
    detach_pre_level_fraction=DETACH_PRE_LEVEL_FRACTION,
    attach_direction=ATTACH_DIRECTION,
    attach_min_points=ATTACH_MIN_POINTS,
    attach_min_monotonicity=ATTACH_MIN_MONOTONICITY,
    attach_min_snr=ATTACH_MIN_SNR,
    attach_min_relative_amplitude=ATTACH_MIN_RELATIVE_AMPLITUDE,
    edge_min_persistence=EDGE_MIN_PERSISTENCE,
    edge_asymmetry_ratio=EDGE_ASYMMETRY_RATIO,
    mode_level_noise_ratio=MODE_LEVEL_NOISE_RATIO,
    mode_level_delta_fraction=MODE_LEVEL_DELTA_FRACTION,
    state_min_confidence=STATE_MIN_CONFIDENCE,
    state_refractory_s=STATE_REFRACTORY_SEC,
    uncertain_margin_s=UNCERTAIN_MARGIN_SEC,
    contact_anchor_window_s=CONTACT_ANCHOR_WINDOW_SEC,
    contact_lowess_window_s=CONTACT_LOESS_WINDOW_SEC,
    contact_robust_iterations=CONTACT_ROBUST_ITERS,
    bslow_tau_s=BSLOW_TAU_SEC,
    bslow_update_rate=BSLOW_UPDATE_RATE,
    bslow_stable_slope_threshold=BSLOW_STABLE_SLOPE_THRESHOLD,
    bslow_stable_offset_threshold=BSLOW_STABLE_OFFSET_THRESHOLD,
    pressure_slope_threshold=PRESSURE_SLOPE_THRESHOLD,
    pressure_offset_threshold=PRESSURE_OFFSET_THRESHOLD,
    j_smooth_window_s=J_SMOOTH_WINDOW_SEC,
)

result2 = decompose_cascade_udeco(udeco2, fs, config)
result3 = decompose_cascade_udeco(udeco3, fs, config)

for name, result in (("Udeco2", result2), ("Udeco3", result3)):
    noncontact_s = np.count_nonzero(result["noncontact_mask"]) / fs
    print(
        f"{name}: 候选 {len(result['edge_candidates'])} 个，"
        f"接受事件 {len(result['J_events'])} 个，Non-contact {noncontact_s:.2f} s"
    )

## 候选边沿与状态机接受事件

候选表用于调参：`edge_width_points` 是平台容差定义的完整边界宽度，`edge_width_10_90_points` 是旧 10%–90% 粗定位宽度，两者并列便于核验；`scale_support` 是达到软门限的支持尺度数。`level_snr`、`classification_reasons` 与 `rejection_reason` 分别反映平台强度、形态筛除和状态机筛除。无可靠两点平台的候选保留中心、边界显示为空；接受事件表只含真正改变状态的 attach/detach。

In [ ]:
CANDIDATE_COLUMNS = [
    "center_s", "start_s", "end_s", "edge_width_points", "edge_width_s",
    "edge_width_10_90_points", "edge_width_10_90_s", "scale_support", "support_scales_s",
    "level_delta", "level_snr", "relative_amplitude", "monotonicity",
    "persistence", "best_scale_s", "edge_score", "direction",
    "candidate_type", "confidence", "accepted",
    "classification_reasons", "rejection_reason",
]


def candidate_table(result):
    '''把内部索引字段整理为便于人工核验的秒制候选表。'''
    rows = []
    for item in result["edge_candidates"]:
        row = dict(item)
        row["center_s"] = row.get("center_index", 0) / fs
        row["start_s"] = row["start_index"] / fs if row.get("start_index") is not None else np.nan
        row["end_s"] = row["end_index"] / fs if row.get("end_index") is not None else np.nan
        row["classification_reasons"] = ", ".join(row.get("classification_reasons", ()))
        rows.append(row)
    frame = pd.DataFrame(rows)
    if frame.empty:
        return pd.DataFrame(columns=CANDIDATE_COLUMNS)
    return frame.reindex(columns=CANDIDATE_COLUMNS).sort_values("center_s")


def event_table(result):
    '''只展示实际驱动一级状态机切换的事件。'''
    frame = candidate_table({"edge_candidates": result["J_events"]})
    if frame.empty:
        return frame
    frame.insert(0, "event_type", [item["event_type"] for item in result["J_events"]])
    return frame


MORPHOLOGY_REASON_LABELS = {
    "detach_width_too_long": "detach 上升沿超过最大点数",
    "detach_low_snr": "detach 平台差 SNR 不足",
    "detach_small_amplitude": "detach 相对幅度不足",
    "detach_short_persistence": "detach 后平台持续性不足",
    "detach_destination_not_separated": "detach 新模态平台分离不足",
    "detach_origin_not_contact_like": "detach 前平台不符合接触参考",
    "attach_width_too_short": "attach 下降沿点数过少",
    "attach_low_snr": "attach 平台差 SNR 不足",
    "attach_small_amplitude": "attach 相对幅度不足",
    "attach_low_monotonicity": "attach 整体下降方向性不足",
    "attach_source_not_separated": "attach 前非接触平台分离不足",
    "edge_near_boundary": "候选靠近记录边界",
    "no_level_change": "前后平台没有持续差异",
    "direction_not_supported": "方向不符合当前物理极性",
}
STATE_REASON_LABELS = {
    "low_confidence": "置信度不足",
    "inside_refractory_period": "处于已接受事件的不应期",
    "attach_not_slower": "attach 未达到相对 detach 的慢速比例",
    "edge_incompatible_with_current_state": "边沿与当前 Contact 状态不相容",
}


def print_edge_filter_stats(name, result):
    '''分别统计形态层和状态机层筛除原因；不改变任何候选或分解结果。'''
    items = result["edge_candidates"]
    print(f"{name}：{len(items)} 个候选，{sum(item['accepted'] for item in items)} 个事件被接受")
    print("  形态类型：")
    print(pd.Series([item["candidate_type"] for item in items], dtype="string").value_counts().to_string())
    morphology = [
        reason for item in items if item["candidate_type"] not in {"detach", "attach"}
        for reason in item.get("classification_reasons", ())
    ]
    state = [
        item["rejection_reason"] for item in items
        if item["candidate_type"] in {"detach", "attach"} and not item["accepted"]
    ]
    for title, reasons, labels in (
        ("形态门槛未通过（同一候选可多因）", morphology, MORPHOLOGY_REASON_LABELS),
        ("状态机拒绝（每个候选单因）", state, STATE_REASON_LABELS),
    ):
        print(f"  {title}：")
        counts = pd.Series(reasons, dtype="string").value_counts()
        print("\n".join(f"    {reason}: {labels.get(reason, reason)} — {count}" for reason, count in counts.items()) or "    无")


for name, result in (("Udeco2", result2), ("Udeco3", result3)):
    candidates = candidate_table(result)
    print_edge_filter_stats(name, result)
    print(f"\n{name} 候选边沿：共 {len(candidates)} 行，显示前 {EDGE_TABLE_MAX_ROWS} 行")
    display(candidates.head(EDGE_TABLE_MAX_ROWS))
    print(f"{name} 状态机接受事件：")
    display(event_table(result))

## 交互诊断图

每一路使用独立 Figure，互不影响；同一路六行默认联动缩放。第三行在 edge_signal = clean − Bcontext 上用同色细线连接候选起止点：上三角表示起点、下三角表示终点；若平台边界未定，只画空心中心点。红色 detach、蓝色 attach、灰色 ambiguous 均为候选而不一定被状态机接受。第四行背景表示一级状态：浅绿 Contact、浅蓝 Non-contact、浅橙 Transition、浅灰 Uncertain。第六行绿色背景表示 `Bslow update enabled`，即该点满足真实稳定 Contact 条件、允许慢基线更新。

候选点可悬停查看边沿宽度、SNR、方向、分类、置信度和拒绝原因。工具栏支持框选、平移、双击复位，鼠标滚轮可缩放。

In [ ]:
EDGE_COLORS = {
    "detach": "#d62728",
    "attach": "#1f77b4",
    "pressure_like": "#ff7f0e",
    "ambiguous": "#7f7f7f",
}


def _plot_range(time_axis):
    '''计算初始显示范围；只改变坐标轴，不裁剪算法数组。'''
    start = float(time_axis[0]) if PLOT_START_SEC is None else float(PLOT_START_SEC)
    end = float(time_axis[-1]) if PLOT_END_SEC is None else float(PLOT_END_SEC)
    return [max(start, float(time_axis[0])), min(end, float(time_axis[-1]))]


def _mask_spans(mask, time_axis, min_points=1, merge_gap_points=0, max_spans=160):
    '''把布尔 mask 压缩为绘图矩形区间，避免逐点添加 shape。'''
    indices = np.flatnonzero(np.asarray(mask, dtype=bool))
    if not indices.size:
        return []
    breaks = np.flatnonzero(np.diff(indices) > merge_gap_points + 1) + 1
    groups = np.split(indices, breaks)
    dt = 1.0 / fs
    spans = [
        (float(time_axis[group[0]]), float(time_axis[group[-1]] + dt))
        for group in groups if len(group) >= min_points
    ]
    return spans[:max_spans]


def _add_mask_regions(fig, mask, time_axis, row, color, opacity, min_points=1, merge_gap=0):
    '''在指定子图添加状态/更新区背景；背景不修改任何数据。'''
    for x0, x1 in _mask_spans(mask, time_axis, min_points, merge_gap):
        fig.add_vrect(
            x0=x0, x1=x1, fillcolor=color, opacity=opacity,
            line_width=0, layer="below", row=row, col=1,
        )


def _add_state_regions(fig, result, time_axis, row):
    '''以互不覆盖的颜色显示状态机输出；Uncertain 叠在最上层。'''
    _add_mask_regions(fig, result["contact_mask"], time_axis, row, "#2ca02c", 0.045, 3, 2)
    _add_mask_regions(fig, result["noncontact_mask"], time_axis, row, "#1f77b4", 0.12, 1, 0)
    _add_mask_regions(fig, result["transition_mask"], time_axis, row, "#ff7f0e", 0.16, 1, 0)
    _add_mask_regions(fig, result["uncertain_mask"], time_axis, row, "#7f7f7f", 0.10, 3, 2)


def _candidate_customdata(items):
    '''构造 Plotly 悬停信息；缺失字段用 NaN/空字符串清楚呈现。'''
    return np.asarray([
        [
            item.get("edge_width_points", np.nan),
            item.get("edge_width_s", np.nan),
            item.get("edge_width_10_90_points", np.nan),
            item.get("level_snr", np.nan),
            item.get("direction", ""),
            item.get("candidate_type", ""),
            item.get("confidence", np.nan),
            item.get("rejection_reason") or "accepted",
            item.get("scale_support", np.nan),
            ", ".join(item.get("classification_reasons", ())),
        ]
        for item in items
    ], dtype=object)


def plot_channel_diagnostics(time_axis, result, channel_name, sample_name):
    '''绘制单路六行交互诊断图；两路分别调用，因此可独立放大。'''
    fig = make_subplots(
        rows=6, cols=1,
        shared_xaxes=LINK_X_AXES_WITHIN_CHANNEL,
        vertical_spacing=0.028,
        specs=[[{}], [{"secondary_y": True}], [{}], [{}], [{}], [{}]],
        subplot_titles=(
            "1. Raw / 3 点 Median clean",
            "2. 低频 Bcontext / 去上下文 edge_signal",
            "3. edge_signal = clean − Bcontext 上的 ambiguous / attach / detach 起止点",
            "4. 多尺度候选边沿与 Contact 状态机",
            "5. 接触模态重建 Ucontact / 最终 Bm_contact",
            "6. Bslow / P / J（绿色背景：允许 Bslow 更新）",
        ),
    )

    fig.add_trace(go.Scatter(x=time_axis, y=result["raw"], name="Raw", line=dict(color="#555", width=1)), row=1, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["clean"], name="Median clean", line=dict(color="#9467bd", width=1)), row=1, col=1)

    fig.add_trace(go.Scatter(x=time_axis, y=result["Bcontext"], name="Bcontext (10 s)", line=dict(color="#2ca02c", width=2)), row=2, col=1, secondary_y=False)
    fig.add_trace(go.Scatter(x=time_axis, y=result["edge_signal"], name="clean-Bcontext", line=dict(color="#7f7f7f", width=1)), row=2, col=1, secondary_y=True)

    # 在算法实际测量形态的 edge_signal 上标注候选边沿；细线连起止点，符号区分起点/终点。
    fig.add_trace(go.Scatter(x=time_axis, y=result["edge_signal"], name="edge_signal / candidate intervals", line=dict(color="#555", width=1)), row=3, col=1)
    for kind in ("ambiguous", "attach", "detach"):
        items = [item for item in result["edge_candidates"] if item.get("candidate_type") == kind]
        # 无可靠前/后平台时只画中心点，不把中心伪装成 start/end。
        unresolved = [item for item in items if item.get("start_index") is None or item.get("end_index") is None]
        if unresolved:
            centers = np.asarray([int(item["center_index"]) for item in unresolved])
            fig.add_trace(go.Scatter(
                x=time_axis[centers], y=result["edge_signal"][centers], mode="markers",
                name=f"{kind} 中心（边界未定）", marker=dict(color=EDGE_COLORS[kind], size=8, symbol="circle-open"),
                customdata=_candidate_customdata(unresolved),
                hovertemplate="%{x:.3f} s / edge_signal=%{y:.5f}<br>边界未定<br>形态原因=%{customdata[9]}<extra></extra>",
            ), row=3, col=1)
        items = [item for item in items if item.get("start_index") is not None and item.get("end_index") is not None]
        if not items:
            continue
        starts = np.asarray([int(item.get("start_index", item["center_index"])) for item in items])
        ends = np.asarray([int(item.get("end_index", item["center_index"])) for item in items])
        starts = np.clip(starts, 0, len(time_axis) - 1)
        ends = np.clip(ends, 0, len(time_axis) - 1)
        segment_x = [point for start, end in zip(starts, ends) for point in (time_axis[start], time_axis[end], None)]
        segment_y = [point for start, end in zip(starts, ends) for point in (result["edge_signal"][start], result["edge_signal"][end], None)]
        fig.add_trace(go.Scatter(x=segment_x, y=segment_y, mode="lines", showlegend=False, hoverinfo="skip", line=dict(color=EDGE_COLORS[kind], width=1.4)), row=3, col=1)
        for phase, indices, symbol in (("起点", starts, "triangle-up"), ("终点", ends, "triangle-down")):
            fig.add_trace(go.Scatter(
                x=time_axis[indices], y=result["edge_signal"][indices], mode="markers",
                name=f"{kind} {phase}", marker=dict(color=EDGE_COLORS[kind], size=7, symbol=symbol),
                customdata=_candidate_customdata(items),
                hovertemplate=("%{x:.3f} s / edge_signal=%{y:.5f}<br>" + kind + " " + phase + "<br>"
                               "完整宽度=%{customdata[0]} 点 / %{customdata[1]:.4f} s；旧10–90%=%{customdata[2]} 点<br>"
                               "支持尺度=%{customdata[8]}；平台 SNR=%{customdata[3]:.2f}<br>方向=%{customdata[4]}<br>"
                               "置信度=%{customdata[6]:.2f}<br>结果=%{customdata[7]}<extra></extra>"),
            ), row=3, col=1)

    _add_state_regions(fig, result, time_axis, row=4)
    fig.add_trace(go.Scatter(x=time_axis, y=result["edge_score"], name="edge_score", line=dict(color="#1f77b4", width=1)), row=4, col=1)
    fig.add_hline(y=EDGE_Z_THRESHOLD, line_dash="dash", line_color="#d62728", row=4, col=1)
    fig.add_hline(y=-EDGE_Z_THRESHOLD, line_dash="dash", line_color="#d62728", row=4, col=1)
    for kind in ("detach", "attach", "pressure_like", "ambiguous"):
        items = [item for item in result["edge_candidates"] if item.get("candidate_type") == kind]
        if not items:
            continue
        centers = np.asarray([int(item.get("center_index", 0)) for item in items])
        centers = np.clip(centers, 0, len(time_axis) - 1)
        fig.add_trace(
            go.Scatter(
                x=time_axis[centers], y=result["edge_score"][centers],
                mode="markers", name=f"candidate: {kind}",
                marker=dict(color=EDGE_COLORS[kind], size=8, symbol="diamond"),
                customdata=_candidate_customdata(items),
                hovertemplate=(
                    "t=%{x:.3f} s<br>edge_score=%{y:.2f}<br>"
                    "完整宽度=%{customdata[0]} 点 / %{customdata[1]:.4f} s；旧10–90%=%{customdata[2]} 点<br>"
                    "支持尺度=%{customdata[8]}；平台 SNR=%{customdata[3]:.2f}<br>方向=%{customdata[4]}<br>"
                    "分类=%{customdata[5]}<br>置信度=%{customdata[6]:.2f}<br>"
                    "形态原因=%{customdata[9]}<br>结果=%{customdata[7]}<extra></extra>"
                ),
            ), row=4, col=1,
        )

    _add_mask_regions(fig, result["ucontact_synthetic_mask"], time_axis, 5, "#17becf", 0.13, 1, 0)
    fig.add_trace(go.Scatter(x=time_axis, y=result["Ucontact"], name="Ucontact", line=dict(color="#17becf", width=1.4)), row=5, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["Bm_contact"], name="Bm_contact", line=dict(color="#ff7f0e", width=2)), row=5, col=1)

    # update mask 往往断续，合并极短间隙并过滤不足 0.1 s 的片段，让绿色背景可读。
    _add_mask_regions(fig, result["bslow_update_mask"], time_axis, 6, "#2ca02c", 0.09, max(2, int(0.10 * fs)), 3)
    fig.add_trace(go.Scatter(x=time_axis, y=result["Bslow"], name="Bslow", line=dict(color="#2ca02c", width=2)), row=6, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["P"], name="P", line=dict(color="#d62728", width=1.5)), row=6, col=1)
    fig.add_trace(go.Scatter(x=time_axis, y=result["J"], name="J", line=dict(color="#9467bd", width=1.5)), row=6, col=1)

    for row in range(1, 7):
        fig.update_xaxes(range=_plot_range(time_axis), title_text="Time (s)" if row == 6 else None, row=row, col=1)
        fig.update_yaxes(title_text="Udeco" if row != 4 else "Robust z-score", row=row, col=1)
    fig.update_yaxes(title_text="clean-Bcontext", row=2, col=1, secondary_y=True)
    fig.update_layout(
        title=f"{sample_name} — {channel_name} 形态边沿与级联状态诊断",
        height=1500, template="plotly_white", hovermode="x unified",
        dragmode="zoom", legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        margin=dict(l=70, r=70, t=125, b=55),
    )
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig


def plot_summary(time_axis, result_u2, result_u3, sample_name):
    '''绘制最终双路摘要；两行 x 轴不联动，便于分别放大核验。'''
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=False, vertical_spacing=0.10,
        subplot_titles=("Udeco2: Raw / clean / J / P / Bslow / residual", "Udeco3: Raw / clean / J / P / Bslow / residual"),
    )
    for row, result in enumerate((result_u2, result_u3), start=1):
        _add_state_regions(fig, result, time_axis, row=row)
        fig.add_trace(go.Scatter(x=time_axis, y=result["raw"], name=f"Raw Udeco{row + 1}", line=dict(color="#555", width=1)), row=row, col=1)
        # clean 是进入分解算法的 4 点中值滤波信号；与 Raw 同图可检查边沿是否被钝化。
        fig.add_trace(go.Scatter(x=time_axis, y=result["clean"], name=f"clean Udeco{row + 1}", line=dict(color="#17becf", width=1.3)), row=row, col=1)
        fig.add_trace(go.Scatter(x=time_axis, y=result["J"], name=f"J{row + 1}", line=dict(color="#9467bd", width=1.5)), row=row, col=1)
        fig.add_trace(go.Scatter(x=time_axis, y=result["P"], name=f"P{row + 1}", line=dict(color="#d62728", width=1.5)), row=row, col=1)
        fig.add_trace(go.Scatter(x=time_axis, y=result["Bslow"], name=f"Bslow{row + 1}", line=dict(color="#2ca02c", width=2)), row=row, col=1)
        # residual 是 clean 扣除 J、P、Bslow 后的剩余项，不是 Raw 的滤波误差。
        fig.add_trace(go.Scatter(x=time_axis, y=result["residual"], name=f"residual{row + 1}", line=dict(color="#8c564b", width=1, dash="dot")), row=row, col=1)
        fig.update_xaxes(range=_plot_range(time_axis), title_text="Time (s)", row=row, col=1)
        fig.update_yaxes(title_text="Udeco", row=row, col=1)
    fig.update_layout(
        title=f"{sample_name} — Udeco 级联状态最终分解",
        height=800, template="plotly_white", hovermode="x unified", dragmode="zoom",
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        margin=dict(l=70, r=40, t=120, b=55),
    )
    fig.show(config={"scrollZoom": True, "displaylogo": False})
    return fig

In [ ]:
# Udeco2 独立交互图：缩放和平移不会改变 Udeco3 的视图。
diagnostic_fig2 = plot_channel_diagnostics(time, result2, "Udeco2", SAMPLE_NAME)

In [ ]:
# Udeco3 独立交互图：重点核验其候选是否被正确保留为压力或慢变，而非伪 Non-contact。
diagnostic_fig3 = plot_channel_diagnostics(time, result3, "Udeco3", SAMPLE_NAME)

## 最终分解摘要

最后一个代码单元只负责显示最终结果。上下两图分别为 Udeco2/Udeco3，每图同时显示 Raw、clean、J、P、Bslow、residual，并叠加 Contact/Non-contact/Transition/Uncertain 状态背景；两行可以分别放大。Raw 未经中值滤波；分解恒等式对应 clean = J + P + Bslow + residual。各分量需要按同一时刻相加，不能期待它们在图上逐条重合。

In [ ]:
summary_fig = plot_summary(time, result2, result3, SAMPLE_NAME)